In [1]:
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/anvarnarz/praktikum_datasets/main/housing_data_08-02-2021.csv')
df.head()

,location,district,rooms,size,level,max_levels,price
0,"город Ташкент, Юнусабадский район, Юнусабад 8-...",Юнусабадский,3,57,4,4,52000
1,"город Ташкент, Яккасарайский район, 1-й тупик ...",Яккасарайский,2,52,4,5,56000
2,"город Ташкент, Чиланзарский район, Чиланзар 2-...",Чиланзарский,2,42,4,4,37000
3,"город Ташкент, Чиланзарский район, Чиланзар 9-...",Чиланзарский,3,65,1,4,49500
4,"город Ташкент, Чиланзарский район, площадь Актепа",Чиланзарский,3,70,3,5,55000


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

url = 'https://raw.githubusercontent.com/anvarnarz/praktikum_datasets/main/housing_data_08-02-2021.csv'
df = pd.read_csv(url)

df['size'] = pd.to_numeric(df['size'], errors='coerce')


df = df[df['price'].apply(lambda x: str(x).isdigit())]
df['price'] = df['price'].astype(float)

df.dropna(inplace=True)

df = df[(df['size'] > 15) & (df['size'] < 500)]
df = df[(df['price'] > 5000) & (df['price'] < 500000)]

X = df[['rooms', 'size', 'level', 'max_levels']]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predicts = model.predict(X_test)
mae = mean_absolute_error(y_test, predicts)

print(f"Model tayyor! O'rtacha xatolik: {mae:.2f} $")

sample = [[3, 70, 4, 5]]
result = model.predict(sample)
print(f"Bashorat qilingan narx: {result[0]:.2f} $")

Model tayyor! O'rtacha xatolik: 12077.29 $
Bashorat qilingan narx: 49847.30 $


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [3]:
!pip install fastapi uvicorn pyngrok nest_asyncio joblib pydantic pandas scikit-learn

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import joblib

url = 'https://raw.githubusercontent.com/anvarnarz/praktikum_datasets/main/housing_data_08-02-2021.csv'
df = pd.read_csv(url)

df['size'] = pd.to_numeric(df['size'], errors='coerce')
df = df[df['price'].apply(lambda x: str(x).isdigit())]
df['price'] = df['price'].astype(float)
df.dropna(inplace=True)

df = df[(df['size'] > 15) & (df['size'] < 500)]
df = df[(df['price'] > 5000) & (df['price'] < 500000)]

X = df[['rooms', 'size', 'level', 'max_levels']]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predicts = model.predict(X_test)
mae = mean_absolute_error(y_test, predicts)
print(f"Model tayyor! O'rtacha xatolik: {mae:.2f} $")

joblib.dump(model, 'tashkent_housing_model.pkl')
print("Model 'tashkent_housing_model.pkl' nomi bilan muvaffaqiyatli saqlandi!")

Model tayyor! O'rtacha xatolik: 12077.29 $
Model 'tashkent_housing_model.pkl' nomi bilan muvaffaqiyatli saqlandi!


In [5]:
%%writefile main.py
from fastapi import FastAPI
from pydantic import BaseModel, Field, model_validator
import joblib
import pandas as pd

app = FastAPI(title="Toshkent uylarining narxini bashorat qilish API")

model = joblib.load('tashkent_housing_model.pkl')

class UyMalumoti(BaseModel):
    rooms: int = Field(..., gt=0, description="Xonalar soni noldan katta bo'lishi shart")
    size: float = Field(..., gt=15, lt=500, description="Uy maydoni 15 va 500 orasida bo'lishi shart")
    level: int = Field(..., ge=1, description="Qavat 1 dan kichik bo'la olmaydi")
    max_levels: int = Field(..., ge=1, description="Binoning umumiy qavatlari soni")

    @model_validator(mode='after')
    def check_level_logic(self):
        if self.level > self.max_levels:
            raise ValueError("Uy joylashgan qavat (level) binoning umumiy qavatlar sonidan (max_levels) katta bo'lishi mumkin emas!")
        return self

@app.get("/")
def home():
    return {"status": "Ishlamoqda", "loyiha": "Toshkent uy bozorini bashorat qilish API"}

@app.post("/predict")
def predict_house_price(uy: UyMalumoti):
    input_data = pd.DataFrame([{
        'rooms': uy.rooms,
        'size': uy.size,
        'level': uy.level,
        'max_levels': uy.max_levels
    }])

    prediction = model.predict(input_data)

    return {
        "kiritilgan_malumotlar": uy,
        "bashorat_qilingan_narx_usd": round(prediction[0], 2)
    }

Writing main.py


In [6]:
!pkill ngrok
!pkill uvicorn
from pyngrok import ngrok
try:
    ngrok.kill()
except:
    pass
print("Tizim to'liq tozalandi! Endi bemalol ishga tushirsak bo'ladi.")

Tizim to'liq tozalandi! Endi bemalol ishga tushirsak bo'ladi.


In [7]:
import subprocess
import time

with open("uvicorn.log", "w") as log_file:
    subprocess.Popen(["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"], stdout=log_file, stderr=log_file)

time.sleep(3)

with open("uvicorn.log", "r") as log_file:
    print(log_file.read())

INFO:     Started server process [1495]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)



In [8]:
from pyngrok import ngrok

NGROK_TOKEN = "38SvKY1NpQe9i2Sjh9byqoRXDN2_7bHu44Zptyy53HgV41x7b"
ngrok.set_auth_token(NGROK_TOKEN)

try:
    public_url = ngrok.connect(8000)
    print("*" * 60)
    print(f"Sizning API manzilingiz: {public_url.public_url}")
    print(f"Swagger UI xujjatlari (Shu linkka kiring): {public_url.public_url}/docs")
    print("*" * 60)
except Exception as e:
    print(f"Xatolik yuz berdi: {e}")

************************************************************
Sizning API manzilingiz: https://retrievable-kayleen-stealthy.ngrok-free.dev
Swagger UI xujjatlari (Shu linkka kiring): https://retrievable-kayleen-stealthy.ngrok-free.dev/docs
************************************************************


In [9]:
%%writefile requirements.txt
fastapi==0.136.3
uvicorn==0.49.0
joblib==1.5.3
pydantic==2.12.3
pandas==2.2.2
scikit-learn==1.6.1

Writing requirements.txt
